# AlphaCluster — Kaggle GPU Training

Train the AlphaCluster RL agent on Kaggle's free GPU (Tesla P100 / T4).

## Prerequisites

Attach two Kaggle Datasets to this notebook:

1. **alphacluster** — the repo (upload the project directory as a zip)
   - Expected at `/kaggle/input/alphacluster/src/`
2. **alphacluster-data** — your parquet data files
   - Expected: `/kaggle/input/alphacluster-data/btcusdt_5m.parquet`
   - Optional: `/kaggle/input/alphacluster-data/btcusdt_funding.parquet`

Enable **GPU accelerator** in notebook settings for faster training.

## Cell 1 — Setup

In [ ]:
!pip install -q "stable-baselines3>=2.4,<3.0" "gymnasium>=0.29,<1.0" pyarrow python-dotenv tqdm rich

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # Suppress TF CUDA registration warnings

import warnings
warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")

import sys
from pathlib import Path

# Add the project source to the Python path
SRC_DIR = "/kaggle/input/datasets/dawidgwczyk/alphacluster-repo/src"
sys.path.insert(0, SRC_DIR)

# Verify the import works
import alphacluster
print(f"AlphaCluster loaded from {Path(alphacluster.__file__).parent}")

# Check GPU availability
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Training will be slower.")

## Cell 2 — Load & Split Data

In [ ]:
import pandas as pd

DATA_DIR = Path("/kaggle/input/datasets/dawidgwczyk/alphacluster-data")
KLINES_PATH = DATA_DIR / "btcusdt_5m.parquet"
FUNDING_PATH = DATA_DIR / "btcusdt_funding.parquet"

assert KLINES_PATH.exists(), f"Kline data not found at {KLINES_PATH}"

klines_df = pd.read_parquet(KLINES_PATH, engine="pyarrow")
print(f"Loaded {len(klines_df):,} candles")
print(f"Date range: {klines_df.iloc[0, 0]} to {klines_df.iloc[-1, 0]}")

funding_df = None
if FUNDING_PATH.exists():
    funding_df = pd.read_parquet(FUNDING_PATH, engine="pyarrow")
    print(f"Loaded {len(funding_df):,} funding records")
else:
    print("No funding data found; training without funding rates.")

# Chronological split: 70% train / 15% val / 15% test
n = len(klines_df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = klines_df.iloc[:train_end].reset_index(drop=True)
val_df = klines_df.iloc[train_end:val_end].reset_index(drop=True)
test_df = klines_df.iloc[val_end:].reset_index(drop=True)

print(f"\nData split: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

## Cell 3 — Create Environments

In [ ]:
from alphacluster.agent.config import TrainingConfig
from alphacluster.env.trading_env import TradingEnv
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor

config = TrainingConfig(total_timesteps=1_000_000)

N_ENVS = 4  # parallel environments for faster rollout collection

def make_train_env(rank: int):
    """Factory that returns a closure for DummyVecEnv."""
    def _init():
        env = TradingEnv(
            df=train_df,
            funding_df=funding_df,
            window_size=config.window_size,
            episode_length=config.episode_length,
        )
        return Monitor(env)
    return _init

print(f"Creating {N_ENVS} parallel training environments ...")
train_env = DummyVecEnv([make_train_env(i) for i in range(N_ENVS)])

eval_env = None
if len(val_df) > config.window_size + config.episode_length:
    print("Creating validation environment ...")
    eval_env = TradingEnv(
        df=val_df,
        funding_df=funding_df,
        window_size=config.window_size,
        episode_length=config.episode_length,
    )
else:
    print("Validation set too small for evaluation; skipping eval callback.")

print(f"\nConfig: lr={config.learning_rate}, batch={config.batch_size}, "
      f"timesteps={config.total_timesteps:,}")
print(f"Parallel envs: {N_ENVS} (rollout collection is {N_ENVS}x faster)")
print(f"Observation space: {train_env.observation_space}")
print(f"Action space: {train_env.action_space}")

## Cell 4 — Train

In [ ]:
from alphacluster.agent.trainer import create_agent, save_agent, train

MODELS_DIR = Path("/kaggle/working/models")
CHECKPOINT_DIR = MODELS_DIR / "checkpoints"

print("Creating PPO agent ...")
agent = create_agent(train_env, config)

print(f"Training for {config.total_timesteps:,} timesteps ...")
print(f"Checkpoints: {CHECKPOINT_DIR}")

agent = train(
    agent=agent,
    config=config,
    eval_env=eval_env,
    checkpoint_dir=str(CHECKPOINT_DIR),
    run_tournament=False,
    progress_bar=False,
)

# Save final model (SB3 zip — may not load on different NumPy versions)
final_path = MODELS_DIR / "ppo_trading_final"
save_agent(agent, final_path)
print(f"\nFinal model saved to {final_path}")

# Save portable PyTorch state dict (works across NumPy 1.x / 2.x)
import torch
portable_path = MODELS_DIR / "ppo_trading_final.pt"
torch.save(agent.policy.state_dict(), portable_path)
print(f"Portable state dict saved to {portable_path}")

## Cell 5 — Evaluate on Test Set

In [ ]:
from alphacluster.backtest.runner import run_backtest
from alphacluster.backtest.metrics import calculate_metrics, print_report

print("Running backtest on test set ...")
test_env = TradingEnv(
    df=test_df,
    funding_df=funding_df,
    window_size=config.window_size,
    episode_length=config.episode_length,
)

result = run_backtest(model=agent, env=test_env, n_episodes=3)
metrics = calculate_metrics(result)
print_report(metrics)

## Cell 6 — Download Instructions

In [ ]:
import os

print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print()
print("Model files in /kaggle/working/ (download from Output tab):")
print()

for root, dirs, files in os.walk("/kaggle/working/models"):
    level = root.replace("/kaggle/working/", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for f in sorted(files):
        size_mb = os.path.getsize(os.path.join(root, f)) / (1024 * 1024)
        print(f"{sub_indent}{f}  ({size_mb:.1f} MB)")

print()
print("To use locally:")
print("  1. Download ppo_trading_final.pt from the Output tab")
print("     (use .pt for cross-platform compatibility, .zip only works")
print("      if local NumPy version matches Kaggle's)")
print("  2. Place it in your local models/ directory")
print("  3. Run: make evaluate-model MODEL_PATH=models/ppo_trading_final.pt")